Name: Shaun Clarke

Course: CSC6314 Deep Learning

Instructor: Margaret Mulhall

Assignment: Project 5

The goal of this assignment is to transition from traditional "rule-based" logic to Modern NLP. You will build a mini-Foundation Model using the nn.Transformer architecture to classify SMS messages as Spam (promotional/malicious) or Ham (normal/safe). By implementing a self-attention mechanism, you will observe how the model learns to weight specific words- like "WIN"-differently based on the words surrounding them.


In [15]:
# Core PyTorch
import torch
# Neural network building blocks (layers and other tools)
import torch.nn as nn
# Base class for custom datasets
from torch.utils.data import Dataset
# DataLoader batches data, shuffles, and handles multiprocessing
from torch.utils.data import DataLoader
from typing import List, Dict, Tuple

In [16]:
# Variables that will be used throughout
# Every sequence will be padded or truncated to this length
MAX_LEN: int = 10
# The dimension of each word's embedding vector
EMBED_DIM: int = 32
# The number of parallel attention heads
N_HEAD: int = 2
# The total number of training epochs
N_EPOCHS: int = 20
# The samples per gradient update
BATCH_SIZE: int = 8
# The Adam Optimizer's learning rate
LR: float = 0.001
# The <PAD> token index fills short sequences
PAD_IDX: int = 0
# The <UNK> token index handles words that are not in the vocabulary
UNK_IDX: int = 1
# The number of output classes. 0=HAM, 1=SPAM
N_CLASSES: int = 2

In [21]:
from IPython.core.displayhook import tokenize
##############################################
# Data Engineering: The Manual Tokenizer
##############################################

# A small sample dataset of raw sms messages
RAW_DATA: list[tuple[str, int]] = [
    ("win cash now", 1),
    ("free money click here", 1),
    ("you won a prize", 1),
    ("urgent claim your reward", 1),
    ("buy now limited offer", 1),
    ("see you later", 0),
    ("can we meet tomorrow", 0),
    ("how are you doing", 0),
    ("dinner at seven tonight", 0),
    ("call me when you are free", 0),
]

# This function builds a word to integer dictionary from the sms message dataset
def build_vocab(data: List[str]) -> Dict[str, int]:
  """
  This function builds a word to integer dictionary from the sms message dataset
  :param data: List of sms messages
  :return: Dictionary of (word, index) pairs
  """
  # Dictionary to hold vocabulary
  vocab: Dict[str, int] = {
      "<PAD>": PAD_IDX, # Index 0 for padding token
      "<UNK>": UNK_IDX, # Index 1 for unknown word token
  }

  # Getting vocabulary from data
  for sentence, _ in data:
    # splitting comments into individual words
    for word in sentence.lower().split():
      # Making sure word doesn't already exist
      if word not in vocab:
        vocab[word] = len(vocab)

  return vocab

# The function converts a sentence string into a fixed length list of integers
def tokenize_and_pad(sentence: str, vocab: Dict[str, int]) -> List[int]:
  """
  The function converts a sentence string into a fixed length list of integers
  :param sentence: Sentence string
  :param vocab: Dictionary of (word, index) pairs
  :return: List of integers
  """
  tokens: list[int] = []

  # Tokenizing the sentence
  words: List[str] = sentence.lower().split()
  # Getting the integer value for each word and ruturning UNK_IDX if the word doesnt exist
  tokens: List[int] = [vocab.get(word, UNK_IDX) for word in words]

  # Truncating the tokens if they are longer tha nmax length
  if len(tokens) > MAX_LEN:
    tokens = tokens[:MAX_LEN]

  # Padding the tokens if they are shorter than nmax length
  while len(tokens) < MAX_LEN:
    tokens.append(PAD_IDX)

  return tokens


# The dataset is small and the model needs many iterations to converege so here we will augment it
# Setting the augment factor to 10 means repeat it this many times
AUGMENT_FACTOR: int = 10
AUGMENTED_DATA: list[tuple[str, int]] = RAW_DATA * AUGMENT_FACTOR

# This class creates a PyTorch dataset wrapper for the  sms samples
class SMSDataset(Dataset):
  """
  This class creates a PyTorch dataset wrapper for the  sms samples
  """
  def __init__(self, data: List[tuple[str, int]], vocab: Dict[str, int]) -> None:
    self.vocab = vocab
    self.samples = []
    for sentence, label in data:
      tokens = tokenize_and_pad(sentence, vocab)
      self.samples.append((tokens, label))

  # This magic method returns the total number of samples
  def __len__(self) -> int:
    return len(self.samples)

  # This method returns one sample as a pytorch tensor
  def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
    tokens, label = self.samples[idx]
    x = torch.tensor(tokens, dtype=torch.long)
    y = torch.tensor(label, dtype=torch.long)

    return x, y






In [22]:
##############################################
# Architecture: The TransformerSpamFilter
##############################################

# This class will be the transformer spam filter
class TransformerSpamFilter(nn.Module):

  def __init__(self, vocab_size: int) -> None:
    super().__init__()

    self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=PAD_IDX)
    self.encoder_layer = nn.TransformerEncoderLayer(d_model=EMBED_DIM, nhead=N_HEAD, batch_first=True)
    self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=1)
    self.fc = nn.Linear(EMBED_DIM, N_CLASSES)

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    # Embedding Tokens
    embedded = self.embedding(x)

    # Passing embedded tokens through transformer encoder
    transformer_output = self.transformer_encoder(embedded)

    #  Average pooling across the sequence diemnsion to get a fixed size representation
    pooled_output = transformer_output.mean(dim=1)

    # Pass the pooled output through the linear layer for classification
    logits = self.fc(pooled_output)


    return logits

In [23]:
##############################################
# The Implementation Logic: Training & Inference
##############################################

# This function runs the manual training loop for N amoun tof epochs
def train(
    model: torch.nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: torch.nn.Module,
) -> None:

  # Switching the model to training mode
  model.train()

  print(f"\n{'Epoch':>8} | {'Avg Loss':>10}")
  print("-" * 22)

  for epoch in range(1, N_EPOCHS + 1):

    total_loss: float = 0.0
    n_batches: int = 0

    # Iterating over the batches in the loader
    for x_batch, y_batch in loader:

      # Clearing accumulated gradients
      optimizer.zero_grad()
      # The forward pass to get predictions
      logits = model(x_batch)
      loss = criterion(logits, y_batch)
      # The backward pass to compute gradients via atuograd
      loss.backward()
      # Updating the weights using the computed gradients
      optimizer.step()
      # Accumulating the loss for the epoch
      total_loss += loss.item()
      n_batches += 1

    if epoch % 5 == 0:
      avg_loss = total_loss / max(n_batches, 1)
      print(f"{epoch:>8} | {avg_loss:>10.4f}")


# This class classifies a new SMS message as SPAM or HAM
def predict(text: str, model: TransformerSpamFilter, vocab: Dict[str, int]) -> str:
  """
  """

  # Tokenizing text and converting it to a vector
  tokens = tokenize_and_pad(text, vocab)
  x = torch.tensor([tokens], dtype=torch.long)

  # Switching the model to eval mode
  model.eval()

  # Running the forward pass without gradient computation because it is not neccessary for eval mode
  with torch.no_grad():
    logits = model(x)
    pred = torch.argmax(logits, dim=1).item()

  # Mapping the pred(index) to a label
  label_map = {0: "HAM", 1: "SPAM"}

  return label_map[pred]


In [25]:
##############################################
# Requirements & Deliverables
##############################################

def main() -> None:
  print("Requirements & Deliverables")
  print(" Step 1: Building vocabulary ")
  vocab: dict[str, int] = build_vocab(RAW_DATA)
  vocab_size: int = len(vocab)
  print(f"  Vocabulary size: {vocab_size} tokens")
  print(f"  Sample entries : {dict(list(vocab.items())[:5])}")

  print("\nStep 2: Creating dataset and dataloader")
  dataset = SMSDataset(AUGMENTED_DATA, vocab)
  loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
  print(f"  Total samples (augmented): {len(dataset)}")
  print(f"  Batches per epoch        : {len(loader)}")

  print("\nStep 3: Building model")
  model     = TransformerSpamFilter(vocab_size=vocab_size)
  criterion = nn.CrossEntropyLoss()     # Combines log-softmax + NLL loss; expects raw logits
  optimizer = torch.optim.Adam(         # Adam: adaptive learning rates per parameter
      model.parameters(), lr=LR
  )
  print(f"Model architecture:\n{model}")

  print("\nStep 4: Training")
  train(model, loader, optimizer, criterion)

  print("\nStep 5: Inference (grading tests)")
  test_cases: list[tuple[str, str]] = [
      ("win cash now",  "SPAM"),
      ("see you later", "HAM"),
  ]
  for text, expected in test_cases:
      result = predict(text, model, vocab)
      status = "yes" if result == expected else "No"
      print(f"  {status} predict('{text}'): {result}  (expected: {expected})")


if __name__ == "__main__":
  main()


Requirements & Deliverables
 Step 1: Building vocabulary 
  Vocabulary size: 36 tokens
  Sample entries : {'<PAD>': 0, '<UNK>': 1, 'win': 2, 'cash': 3, 'now': 4}

Step 2: Creating dataset and dataloader
  Total samples (augmented): 100
  Batches per epoch        : 13

Step 3: Building model
Model architecture:
TransformerSpamFilter(
  (embedding): Embedding(36, 32, padding_idx=0)
  (encoder_layer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
    )
    (linear1): Linear(in_features=32, out_features=2048, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=2048, out_features=32, bias=True)
    (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): Tr